# 🐾 PawAI

**An Intelligent Multi-Agent System for Comprehensive Dog Care**

## Architecture

```
User Question
     ↓
API Gateway / Router  (LLM JSON plan)
     ↓
Agent Orchestration Layer
     ↓
Agent Pool  ←→  LLM (Llama-3.2-3B) + RAG  ←→  FAISS Vector DB
  ├─ Health Agent    (PubMed veterinary research)
  ├─ Nutrition Agent (AAFCO guidelines + nutrition research)
  └─ Training Agent  (Applied Animal Behaviour Science)
     ↓
Synthesizer → Final Output
```

This notebook implements a comprehensive multi-agent system for dog care that combines RAG (Retrieval-Augmented Generation), LLM inference, and specialized agents for health, nutrition, and training domains.

## Section 1: Environment Setup & Configuration

Configure Google Colab environment variables, mount Google Drive, and upload required modules.

In [ ]:
!pip install -q huggingface-hub transformers torch sentence-transformers faiss-cpu biopython langchain langchain-community langchain-core langchain-text-splitters flask flask-cors streamlit pyngrok requests numpy python-dotenv

print("✅ All dependencies installed successfully!")


In [ ]:
import os

from google.colab import userdata
os.environ['HF_TOKEN']   = userdata.get('HF_TOKEN')
os.environ['NCBI_EMAIL'] = userdata.get('NCBI_EMAIL')

os.environ['CORPUS_DIR']  = '/content/corpus'
os.environ['INDEX_DIR']   = '/content/faiss_indexes'

print('Environment configured.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import files
print('Upload: config.py, data_ingestion.py, rag_pipeline.py, llm.py, agents.py, orchestrator.py, app.py,streamlit_app_ngrok.py')
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

## Section 2: Data Ingestion & Corpus Loading

Load and inspect health, nutrition, and training corpora from JSON files.

In [ ]:
import os, json, warnings
warnings.filterwarnings('ignore')

import config          

from data_ingestion import build_all_corpora, load_corpus
import config
for name, path in [
    ('Health',    config.HEALTH_CORPUS),
    ('Nutrition', config.NUTRITION_CORPUS),
    ('Training',  config.TRAINING_CORPUS),
]:
    docs = load_corpus(path)
    print(f'{name:12s}: {len(docs):4d} chunks  →  {path}')

In [ ]:
import json

for name, path in [
    ('Health',    config.HEALTH_CORPUS),
    ('Nutrition', config.NUTRITION_CORPUS),
    ('Training',  config.TRAINING_CORPUS),
]:
    print(f'\n{'='*60}')
    print(f'  {name} corpus – first 2 chunks')
    print(f'{'='*60}')
    with open(path) as f:
        for i, line in enumerate(f):
            if i >= 2: break
            obj = json.loads(line)
            print(f"doc_id : {obj['doc_id']}")
            print(f"source : {obj.get('source', 'N/A')}")
            print(f"title  : {obj.get('title', 'N/A')}")
            print(f"text   : {obj['text'][:250]}...")
            print()

## Section 3: RAG Pipeline & Vector Database Initialization

Initialize the RAGPipeline and build FAISS indexes for all domains.

In [ ]:
from rag_pipeline import RAGPipeline

rag = RAGPipeline()
rag.build_indexes(force_rebuild=False)
print('\n✅ All FAISS indexes ready.')

In [ ]:
test_queries = [
    ('health',    'my dog is vomiting and lethargic'),
    ('nutrition', 'how much should I feed my Labrador puppy'),
    ('training',  'my dog barks excessively when left alone'),
]

for domain, query in test_queries:
    docs = rag.retrieve(query, domain=domain, k=2)
    rag.show_evidence(query, docs)

## Section 4: LLM Model Loading

Load the Llama-3.2-3B model with 4-bit quantization.

In [ ]:
import llm

llm.load_model(use_4bit=True)   
print(f'\nLoaded model: {llm.get_model_name()}')

response = llm.generate([
    {'role': 'system', 'content': 'You are a dog care assistant.'},
    {'role': 'user',   'content': 'In one sentence, what is your role?'},
])
print('LLM response:', response)

## Section 5: Agent Initialization & Dog Profile Setup

Create DogProfile instances and initialize specialized agents.

In [ ]:
from agents import DogProfile, HealthAgent, NutritionAgent, TrainingAgent

my_dog = DogProfile(
    name           = 'Buddy',
    breed          = 'Labrador Retriever',
    age_years      = 2.0,
    weight_kg      = 30.0,
    sex            = 'male',
    neutered       = True,
    allergies      = ['chicken'],
    health_issues  = [],
    activity_level = 'high',
)

print('Dog profile:', my_dog.summary())

## Section 6: Individual Agent Testing

Test each agent (Health, Nutrition, Training) independently.

In [ ]:
### Health Agent Test

health_agent = HealthAgent(rag)

result = health_agent.run(
    question     = 'Buddy vomited twice this morning and seems tired. Should I be worried?',
    dog          = my_dog,
    show_evidence = True,
)

print('HEALTH AGENT ANSWER')
print(result['answer'])
print(f"\n{result['triage_label']}")

In [ ]:
### Nutrition Agent Test

nutrition_agent = NutritionAgent(rag)

result = nutrition_agent.run(
    question      = 'What should I feed Buddy daily? He has a chicken allergy.',
    dog           = my_dog,
    show_evidence = True,
)

print('NUTRITION AGENT ANSWER')
print(result['answer'])

ci = result['calorie_info']
print(f"\nCalorie Estimate: RER={ci['RER_kcal_per_day']} kcal/day | MER={ci['MER_kcal_per_day']} kcal/day")

In [ ]:
### Training Agent Test

training_agent = TrainingAgent(rag)

result = training_agent.run(
    question      = 'Buddy pulls hard on the leash during walks. How do I train him?',
    dog           = my_dog,
    show_evidence = True,
)

print('TRAINING AGENT ANSWER')
print(result['answer'])

## Section 7: Query Routing & Multi-Agent Orchestration

Test the router to demonstrate multi-agent plan generation for complex questions.

In [ ]:
from orchestrator import route_question

test_questions = [
    'My dog is limping after a run.',
    'What should my senior Beagle eat?',
    'My puppy bites everyone.',
    'My overweight Labrador keeps stealing food and pulls on the leash.',
    'My dog ate chocolate – is this an emergency?',
]

print('ROUTER TESTS\n')
for q in test_questions:
    plan = route_question(q)
    agents_selected = [s['agent'] for s in plan['plan']]
    print(f"Q: {q}")
    print(f"→ {agents_selected}\n")

## Section 8: Integration Tests & Scenario Validation

Execute end-to-end tests with diverse dog profiles and scenarios.

In [ ]:
from orchestrator import run_agent

result = run_agent(
    question     = 'My dog just ate a whole bar of dark chocolate. What should I do?',
    rag          = rag,
    dog          = my_dog,
    show_evidence = True,
)

assert isinstance(result['final_answer'], str) and len(result['final_answer']) > 50
assert result.get('triage') in {'HOME_CARE', 'MONITOR', 'VET_VISIT', 'EMERGENCY'}
print('\n✅ Test 1 passed')

In [ ]:
### Test 2: Puppy Nutrition

puppy = DogProfile(
    name='Luna', breed='Golden Retriever', age_years=0.4,
    weight_kg=8.0, sex='female', neutered=False,
    activity_level='high'
)

result = run_agent(
    question     = 'How many times a day should I feed my 4-month-old Golden Retriever puppy and how much?',
    rag          = rag,
    dog          = puppy,
    show_evidence = True,
)

assert 'nutrition' in [s['agent'] for s in result['plan']]
print('\n✅ Test 2 passed')

In [ ]:
### Test 3: Rescue Dog Training

rescue = DogProfile(
    name='Max', breed='German Shepherd mix', age_years=3.0,
    weight_kg=28.0, sex='male', neutered=True,
    health_issues=['anxiety'],
    activity_level='moderate'
)

result = run_agent(
    question     = 'My rescue dog destroys furniture when I leave the house. How do I stop this?',
    rag          = rag,
    dog          = rescue,
    show_evidence = True,
)

assert 'training' in [s['agent'] for s in result['plan']]

In [ ]:
### Test 4: Senior Dog Multi-Agent Query

senior = DogProfile(
    name='Bella', breed='Beagle', age_years=9.0,
    weight_kg=16.0, sex='female', neutered=True,
    health_issues=['arthritis', 'mild obesity'],
    activity_level='low'
)

result = run_agent(
    question     = 'My 9-year-old arthritic Beagle is overweight and very stubborn on walks. How should I manage her diet and exercise?',
    rag          = rag,
    dog          = senior,
    show_evidence = True,
)

print('\nAgents activated:', [s['agent'] for s in result['plan']])
assert len(result['plan']) >= 2, 'Expected multiple agents for this query'

## Section 9: Evaluation - Triage Accuracy & Personalization

Evaluate triage accuracy and test breed-specific personalization.

In [ ]:
### Triage Accuracy Tests

import re
from orchestrator import run_agent
from agents import DogProfile

eval_dog = DogProfile(
    name='TestDog', breed='Mixed', age_years=4.0,
    weight_kg=20.0, sex='male', neutered=True,
    activity_level='moderate',
)

SCENARIO_TESTS = [
    ('My dog just ate xylitol-containing gum.',      'EMERGENCY', ['poison', 'emergency', 'vet']),
    ('My dog vomited once but seems fine now.',      'MONITOR',   ['monitor', 'watch', 'bland']),
    ('My dog has been limping for 3 days.',          'VET_VISIT', ['vet', 'veterinarian', 'appointment']),
    ('My dog keeps scratching his ears and shaking his head.', 'VET_VISIT', ['ear', 'vet']),
    ('My dog collapsed and his gums look pale.',     'EMERGENCY', ['emergency', 'immediately', 'urgent']),
]

print('EVALUATION – TRIAGE ACCURACY & SAFETY')

correct_triage     = 0
has_disclaimer     = 0
emergency_recall   = 0
n_emergency        = sum(1 for _, t, _ in SCENARIO_TESTS if t == 'EMERGENCY')

results_log = []

for question, expected_triage, keywords in SCENARIO_TESTS:
    result = run_agent(question, rag=rag, dog=eval_dog, show_evidence=False)
    answer = result['final_answer'].lower()
    triage = result.get('triage', 'UNKNOWN')

    triage_ok = triage == expected_triage
    if triage_ok:
        correct_triage += 1

    if expected_triage == 'EMERGENCY' and triage == 'EMERGENCY':
        emergency_recall += 1

    disclaimer_ok = any(w in answer for w in ['veterinarian', 'vet', 'not a licensed', 'consult'])
    if disclaimer_ok:
        has_disclaimer += 1

    kw_found = [k for k in keywords if k in answer]

    status = '✅' if triage_ok else '❌'
    print(f"{status} Triage: {triage:12s} (expected: {expected_triage:12s}) | Q: {question[:55]}...")
    results_log.append({'question': question, 'expected': expected_triage,
                         'got': triage, 'correct': triage_ok})

n = len(SCENARIO_TESTS)
print(f'\nSummary')
print(f'Triage accuracy       : {correct_triage}/{n} = {correct_triage/n*100:.0f}%')
print(f'Disclaimer present    : {has_disclaimer}/{n} = {has_disclaimer/n*100:.0f}%')
print(f'Emergency recall      : {emergency_recall}/{n_emergency} = {emergency_recall/n_emergency*100:.0f}%')

In [ ]:
### Personalization Tests

print('\nEVALUATION – PERSONALISATION (breed / allergy awareness)')

dalmatian = DogProfile(
    name='Spot', breed='Dalmatian', age_years=5.0,
    weight_kg=25.0, sex='male', neutered=True,
    activity_level='high',
)

result = run_agent(
    'What diet is best for my dog?',
    rag=rag, dog=dalmatian, show_evidence=False,
)
answer = result['final_answer'].lower()

personalised = any(w in answer for w in ['dalmatian', 'urate', 'purine', 'low-purine'])
print(f"Dalmatian-specific advice found: {'✅ Yes' if personalised else '❌ No'}")
print(f"\nAnswer excerpt: {result['final_answer'][:400]}...")

# Allergy-aware test
chicken_allergic = DogProfile(
    name='Rosie', breed='Poodle', age_years=3.0,
    weight_kg=10.0, sex='female', neutered=True,
    allergies=['chicken', 'wheat'],
    activity_level='moderate'
)
result2 = run_agent(
    'Suggest a meal plan for my dog.',
    rag=rag, dog=chicken_allergic, show_evidence=False,
)
answer2 = result2['final_answer'].lower()
allergy_aware = 'chicken' in answer2 and any(w in answer2 for w in ['avoid', 'allerg', 'sensitive'])
print(f"\nAllergy awareness (avoid chicken): {'✅ Yes' if allergy_aware else '❌ No'}")

## Section 10: Deployment - Flask & Streamlit Setup

Configure ngrok tunneling, and launch the application.

In [ ]:
### Configure ngrok Tunnel

os.environ['NGROK_AUTH_TOKEN'] = userdata.get('NGROK_AUTH_TOKEN')
from pyngrok import ngrok, conf
if NGROK_AUTH_TOKEN:
    conf.get_default().auth_token = NGROK_AUTH_TOKEN
    print("✅ ngrok token set")
else:
    print("⚠️  No token — add one above!")

In [ ]:
### Launch Flask Backend

import subprocess, sys, time, os

os.system("pkill -f 'app.py'")
time.sleep(1)

flask_proc = subprocess.Popen(
    [sys.executable, "/content/app.py"],
    stdout=open("/content/flask.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(60)

import requests as _req
try:
    r = _req.get("http://localhost:8000/health", timeout=10)
    print(f"✅ Flask is up: {r.json()}")
except Exception as e:
    print(f"⚠️  Not ready yet: {e} — check logs: !cat /content/flask.log")

In [ ]:
### Create Flask Public URL

from pyngrok import ngrok

ngrok.kill()
flask_tunnel = ngrok.connect(8000)
FLASK_PUBLIC_URL = flask_tunnel.public_url
print(f"🔗 Flask public URL: {FLASK_PUBLIC_URL}")

In [ ]:
### Verify Flask Health

import requests as _req
try:
    r = _req.get("http://localhost:8000/health", timeout=10)
    print(f"✅ Flask health check: {r.json()}")
except Exception as e:
    print(f"❌ Flask is DOWN: {e}")
    print("   Re-run previous cell to restart Flask, then re-run this cell.")
    raise SystemExit

In [ ]:
### Launch Streamlit Frontend

import subprocess, sys, time, os
from pyngrok import ngrok

os.system("pkill -f streamlit")
time.sleep(1)

streamlit_proc = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run",
        "/content/streamlit_app_ngrok.py",
        "--server.port=8501",
        "--server.headless=true",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false",
    ],
    stdout=open("/content/streamlit.log", "w"),
    stderr=subprocess.STDOUT,
)

time.sleep(5)
streamlit_tunnel = ngrok.connect(8501)

print(f"\n✅ Your Dog AI Agent is LIVE at:")
print(f"   👉 {streamlit_tunnel.public_url}\n")
print("Flask logs:     !cat /content/flask.log")
print("Streamlit logs: !cat /content/streamlit.log")

In [ ]:
### View Deployment Logs

!cat /content/flask.log

In [ ]:
!cat /content/streamlit.log